# 29 · CRAG / Self-RAG / Adaptive RAG

> Agentic RAG 的三位重要成员：一个**纠偏**、一个**自省**、一个**路由**。

**本文件覆盖知识点**：CRAG(Corrective RAG) / Retrieval Grader / Web Search / Self-RAG / Retrieval Decision·Relevance·Support·Critique / Adaptive RAG

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


In [ ]:
# ===== 本课共用：真实检索底座 =====
# 真语料(data/) → 真切分 → 真向量(text-embedding-v3) → 真索引(FAISS + BM25)
# → 真重排(qwen3-rerank) → 真生成(qwen-plus)。各课在这个底座上演示自己的知识点。
#
# 说明：向量按内容哈希缓存在 .cache/emb.npz（首次真调、之后复用，避免反复花 token）。
# 没配 DASHSCOPE_API_KEY 时仍可用：向量直接从缓存读（是此前真实调用的结果），
# 但需要现场调用模型的重排/生成会打印录制结果并提示配置方式。
from dotenv import load_dotenv; load_dotenv()
import os, re, json, time, hashlib
from pathlib import Path
import numpy as np

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY
_DATA = Path('data') if Path('data').is_dir() else Path.cwd() / 'data'
_CACHE_FILE = Path('.cache') / 'emb.npz'
EMBED_MODEL = 'text-embedding-v3'
RERANK_MODEL = 'qwen3-rerank'
NO_KEY_TIP = ('未配置 DASHSCOPE_API_KEY：需要现场调用模型的部分将展示此前真实调用的录制结果，'
              '在项目根 .env 配置后自动变为实时调用。')

def recorded(text, note=''):
    """无 Key 时展示「此前真实运行的录制结果」。内容来自真实调用，不是编造的假数据。"""
    print(NO_KEY_TIP)
    print('—— 录制结果%s ——' % ('（' + note + '）' if note else ''))
    print(text)

if not _HAS_KEY:
    print(NO_KEY_TIP)

# ---------- 1) 语料：读 data/ 全部 Markdown，按小节切块 ----------
# 注意：评测集*.md 是「人工标注的答案」，不能进索引 —— 否则第 34 课评测时，
# 标注本身会被检索命中，指标虚高（数据泄漏）。这里按文件名前缀排除（含第 33 课产出的 评测集_v2.md）。
_EXCLUDE_PREFIX = '评测集'

def load_chunks(chunk_size=300, overlap=60):
    """按「## 小节」切分，小节过长再按句子窗口滑切。返回 [{'i','text','source','section'}]"""
    out = []
    for p in sorted(_DATA.glob('*.md')):
        if p.name.startswith(_EXCLUDE_PREFIX):
            continue
        section, buf = p.stem, []
        for line in p.read_text(encoding='utf-8').splitlines():
            if line.startswith('## '):
                if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
                section, buf = line[3:].strip(), [line]
            elif line.startswith('# '):
                section = line[2:].strip()
            else:
                buf.append(line)
        if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
    for i, c in enumerate(out):
        c['i'] = i
    return out

def _split_section(lines, section, source, chunk_size, overlap):
    """小节内容按句号聚合成 ~chunk_size 字的片段，相邻片段留 overlap 字重叠"""
    text = '\n'.join(lines).strip()
    if not text: return []
    sents = [s for s in re.split(r'(?<=[。！？\n])', text) if s.strip()]
    chunks, buf = [], ''
    for s in sents:
        if len(buf) + len(s) > chunk_size and buf:
            chunks.append(buf.strip())
            buf = buf[-overlap:] + s          # 保留尾部 overlap 字做上下文重叠
        else:
            buf += s
    if buf.strip(): chunks.append(buf.strip())
    return [{'text': c, 'source': source, 'section': section} for c in chunks]

# ---------- 2) 向量：真调 text-embedding-v3（分批 + 重试 + 内容哈希缓存）----------
def _load_cache():
    if not _CACHE_FILE.exists():
        return {}
    try:
        z = np.load(_CACHE_FILE, allow_pickle=False)
        return dict(zip(z['hashes'].tolist(), z['vectors']))
    except Exception as e:                      # 文件损坏（例如多进程同时写）：当空缓存重建，别让 notebook 挂掉
        print('向量缓存不可读(%s: %s)，将重新向量化：%s' % (type(e).__name__, e, _CACHE_FILE))
        return {}

def _save_cache(cache):
    """写盘前先与磁盘上已有内容合并，再原子替换 —— 避免多个进程同时跑时互相覆盖 / 写坏文件"""
    _CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    for k, v in _load_cache().items():
        cache.setdefault(k, v)
    hs = np.array(list(cache.keys()))
    vs = np.array([cache[h] for h in cache.keys()], dtype='float32')
    # 进程号唯一，别抢同一个临时文件；注意 np.savez_compressed 会自动补 .npz 后缀，临时名必须也是 .npz 结尾
    tmp = _CACHE_FILE.with_name('%s.%d.tmp.npz' % (_CACHE_FILE.stem, os.getpid()))
    np.savez_compressed(tmp, hashes=hs, vectors=vs)
    try:
        os.replace(tmp, _CACHE_FILE)            # 原子替换：别的进程读到的永远是完整文件
    except OSError:                             # 目标被占用时稍等再试
        time.sleep(0.2); os.replace(tmp, _CACHE_FILE)

def _key(text, model):
    return hashlib.sha1((model + '\x00' + text).encode('utf-8')).hexdigest()[:16]

def embed(texts, model=EMBED_MODEL, batch=10):
    """真调 Embedding；命中缓存则直接用（缓存来自真实调用）。返回已 L2 归一化的向量"""
    if isinstance(texts, str): texts = [texts]
    cache, todo = _load_cache(), []
    for t in texts:
        k = _key(t, model)
        if k not in cache and k not in [x[0] for x in todo]:
            todo.append((k, t))
    if todo and not _HAS_KEY:
        raise RuntimeError('本地缓存缺少 %d 条向量，且未配置 DASHSCOPE_API_KEY，无法现场向量化。'
                           '请在项目根 .env 配置 Key 后重跑。' % len(todo))
    if todo:
        from dashscope import TextEmbedding
        pending = todo
        while pending:                                  # 批次过大就减半重试
            b = pending[:batch]
            r = TextEmbedding.call(model=model, input=[t for _, t in b], api_key=_KEY)
            if r.status_code == 200:
                for (k, _), e in zip(b, sorted(r.output['embeddings'], key=lambda e: e['text_index'])):
                    cache[k] = np.array(e['embedding'], dtype='float32')
                pending = pending[len(b):]
            elif batch > 1:
                batch //= 2
            else:
                raise RuntimeError('向量化失败: %s %s' % (r.code, r.message))
        _save_cache(cache)
    v = np.array([cache[_key(t, model)] for t in texts], dtype='float32')
    return v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-10)

# ---------- 3) 索引：FAISS（归一化后内积=余弦）+ BM25 ----------
import faiss
from rank_bm25 import BM25Okapi

def tokenize(text):
    """中文用「单字 + 相邻双字」切词，无需外部分词器（与第 16 课一致）"""
    t = re.sub(r'\s+', '', text)
    return [t[i] for i in range(len(t))] + [t[i:i + 2] for i in range(len(t) - 1)]

CHUNKS = load_chunks()
VECS = embed([c['text'] for c in CHUNKS])
INDEX = faiss.IndexFlatIP(VECS.shape[1]); INDEX.add(VECS)
BM25 = BM25Okapi([tokenize(c['text']) for c in CHUNKS])
print('语料就绪：%d 篇文档 → %d 个片段，向量维度 %d' % (len({c['source'] for c in CHUNKS}), len(CHUNKS), VECS.shape[1]))

# ---------- 4) 检索：稠密 / 稀疏 / 混合（RRF 融合）----------
def dense_retrieve(query, k=5):
    sims, ids = INDEX.search(embed(query), k)
    return [dict(CHUNKS[i], score=float(s), from_='dense') for i, s in zip(ids[0], sims[0]) if i != -1]

def sparse_retrieve(query, k=5):
    scores = BM25.get_scores(tokenize(query))
    top = np.argsort(-scores)[:k]
    return [dict(CHUNKS[i], score=float(scores[i]), from_='bm25') for i in top if scores[i] > 0]

def hybrid_retrieve(query, k=5, rrf_k=60, pool=10):
    """RRF 融合：score = Σ 1/(rrf_k + rank)，只用名次不用原始分数，天然可比"""
    fused = {}
    for name, hits in (('dense', dense_retrieve(query, pool)), ('bm25', sparse_retrieve(query, pool))):
        for rank, h in enumerate(hits, 1):
            cur = fused.setdefault(h['i'], dict(h, score=0.0, from_=set()))
            cur['score'] += 1.0 / (rrf_k + rank)
            cur['from_'].add(name)
    return sorted(fused.values(), key=lambda x: -x['score'])[:k]

# ---------- 5) 重排：真调 DashScope TextReRank ----------
def rerank(query, docs, top_n=3, model=RERANK_MODEL):
    """docs 可以是字符串列表或检索结果 dict 列表；返回 [(文档, 相关性分数)]"""
    texts = [d['text'] if isinstance(d, dict) else d for d in docs]
    if not texts: return []
    if not _HAS_KEY:
        print(NO_KEY_TIP); return [(t, None) for t in texts[:top_n]]
    from dashscope import TextReRank
    r = TextReRank.call(model=model, query=query, documents=texts,
                        top_n=min(top_n, len(texts)), return_documents=False, api_key=_KEY)
    if r.status_code != 200:
        raise RuntimeError('重排失败: %s %s' % (r.code, r.message))
    return [(texts[it['index']], float(it['relevance_score'])) for it in r.output['results']]

# ---------- 6) 生成：qwen-plus（带重试）+ 结构化 JSON 输出 ----------
def chat(prompt, system='你是严谨的 RAG 助手：只依据给定资料回答，资料里没有的就直说不知道。',
         temperature=0.3, model='qwen-plus', retries=3):
    if not _HAS_KEY:
        return None
    from dashscope import Generation
    for attempt in range(retries):
        r = Generation.call(model=model, messages=[{'role': 'system', 'content': system},
                                                   {'role': 'user', 'content': prompt}],
                            temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            return r.output.choices[0].message.content
        if attempt == retries - 1:
            raise RuntimeError('生成失败: %s %s' % (r.code, r.message))
        time.sleep(1.5 * (attempt + 1))          # 限流类错误退避重试
    return None

def chat_json(prompt, system='只输出 JSON，不要任何解释或代码块标记。', retries=2, **kw):
    """要求模型输出 JSON 并解析；解析失败时把报错回喂再试一次"""
    for attempt in range(retries + 1):
        out = chat(prompt, system=system, **kw)
        if out is None: return None
        seg = out[out.find('{'): out.rfind('}') + 1]     # 容忍 ```json 包裹与前后废话
        try:
            return json.loads(seg)
        except Exception as e:
            if attempt == retries: raise
            prompt = prompt + '\n\n上次输出无法解析(%s)，请只输出合法 JSON。' % e
    return None


## 1. CRAG：先评测检索，不好就纠正

```text
Retrieve → Evaluate Retrieval(Retrieval Grader)
                 │
            好? ├─ Yes → Generate
                 └─ No  → Correct：改写查询重试 / 触发 Web Search 补外部知识
```

- **Retrieval Grader**：让 LLM 给“检索到的片段是否与问题相关”打分（0/1 或分级）；
- 相关度高 → 直接用；部分相关 → 合并；不相关 → **纠偏**（重写查询再试，或联网）。

In [ ]:
# CRAG 的真实实现：真实召回 → 模型判官分三档 → 三档走不同纠偏分支
# 与桩版的区别：doc 来自 hybrid_retrieve 的真实召回（不再写死两条），判官由 qwen-plus 输出
# 结构化 JSON（不是关键词规则），分档结果真的决定走哪条分支。
# 三档语义与本课正文一致：正确→直接用；模糊→补充检索后合并；错误→改写查询重来。
import json as _json

_GRADE_SYS = ('你是 CRAG 的检索质量判官。只依据给定片段与问题判断检索质量，输出一个 JSON 对象：'
              '{"quality": "correct|ambiguous|incorrect", "reason": "一句话中文理由"}。'
              'correct=片段足以回答问题；ambiguous=部分相关但信息不足；incorrect=片段与问题无关。'
              '只输出 JSON，不要任何解释。')

_REWRITE_SYS = ('你是 CRAG 的查询改写器。原检索质量不达标，请改写检索词以更好命中知识库。'
                '只输出一个 JSON 对象：{"query": "<改写后的检索词>", "reason": "<一句话理由>"}。禁止其它内容。')

def _fmt(docs):
    return '\n'.join('[%d]（%s · %s）%s' % (i, d['source'], d['section'], d['text'][:220])
                     for i, d in enumerate(docs, 1))

class CRAG:
    """Corrective RAG：判官给检索质量分档，每档一个纠偏动作"""

    def __init__(self, k=4):
        self.k = k

    def retrieve(self, q):
        return hybrid_retrieve(q, k=self.k)        # 真实混合召回（稠密 + BM25 + RRF）

    def grade(self, q, docs):
        """Retrieval Grader：真调模型给三档结论"""
        return chat_json('问题：%s\n\n检索到的片段：\n%s\n\n这些片段的检索质量属于哪一档？' % (q, _fmt(docs)),
                         system=_GRADE_SYS, temperature=0.1)

    def rewrite(self, q, g):
        """模糊/错误档：让模型改写检索词"""
        return chat_json('问题：%s\n判官结论：%s（%s）\n请给出更容易命中的检索词。'
                         % (q, (g or {}).get('quality'), (g or {}).get('reason')),
                         system=_REWRITE_SYS, temperature=0.2)

    def generate(self, q, docs):
        return chat('资料：\n%s\n\n问题：%s\n只依据资料作答；资料不足以回答时明确说明「知识库中没有相关信息」。'
                    % (_fmt(docs), q))

    def run(self, q):
        print('=' * 72)
        print('问题：', q)
        docs = self.retrieve(q)
        print('① 真实召回 %d 条：' % len(docs))
        for d in docs:
            print('   [%s · %s] %s' % (d['source'], d['section'], d['text'][:38].replace('\n', ' ')))
        g = self.grade(q, docs)
        quality = (g or {}).get('quality')
        print('② 判官分诊：', _json.dumps(g, ensure_ascii=False))
        if quality == 'correct':
            print('③ 分支【正确】→ 片段直接用，直接生成（不纠偏）')
            print('答案：', self.generate(q, docs))
        elif quality == 'ambiguous':
            print('③ 分支【模糊】→ 补充检索：改写检索词再补一轮，与原片段合并后生成')
            rw = self.rewrite(q, g)
            print('   改写检索词：', _json.dumps(rw, ensure_ascii=False))
            extra = self.retrieve(rw.get('query') or q) if rw else []
            for d in extra:
                print('   + 补充命中 [%s · %s] %s' % (d['source'], d['section'], d['text'][:38].replace('\n', ' ')))
            merged = list({d['i']: d for d in docs + extra}.values())   # 按片段 id 去重后合并
            print('   合并后 %d 条（原有 %d + 补充 %d）' % (len(merged), len(docs), len(extra)))
            print('答案：', self.generate(q, merged))
        elif quality == 'incorrect':
            print('③ 分支【错误】→ 改写查询重来：换检索词重新召回，并对新结果重新分诊')
            rw = self.rewrite(q, g)
            print('   改写检索词：', _json.dumps(rw, ensure_ascii=False))
            docs2 = self.retrieve(rw.get('query') or q) if rw else []
            for d in docs2:
                print('   + 重新召回 [%s · %s] %s' % (d['source'], d['section'], d['text'][:38].replace('\n', ' ')))
            g2 = self.grade(q, docs2)
            print('   重新分诊：', _json.dumps(g2, ensure_ascii=False))
            if (g2 or {}).get('quality') == 'correct':
                print('   → 纠偏后达标，用新片段生成')
            else:
                print('   → 纠偏后仍不达标，走兜底：如实告知知识库无相关信息（不硬答，防幻觉）')
            print('答案：', self.generate(q, docs2))
            quality = (g2 or {}).get('quality')
        else:
            print('③ 判官未返回可用结论，按兜底处理（不改写、不硬答）')
        return quality

_QUERIES = [
    '星云企业版私有化部署的最低硬件配置是什么？',
    '星云智能客服机器人对比其他厂商有什么优势？',
    '星云机器人支持哪些大模型？',
    '今天上海的天气怎么样？',          # 知识库完全覆盖不到的问题：判官应判 incorrect → 改写查询重来
]

if not _HAS_KEY:
    recorded(r"""========================================================================
问题： 星云企业版私有化部署的最低硬件配置是什么？
① 真实召回 4 条：
   [星云智能产品手册.md · 部署方式] ## 部署方式 产品支持公有云 SaaS 与私有化两种部署方式。公有云版本开通可
   [星云客服FAQ.md · 部署相关] ## 部署相关 **Q：能私有化部署吗？** A：能。标准版及以上可申请私有
   [计费与SLA.md · 私有化部署计费] ## 私有化部署计费 私有化部署为一次性授权费 + 年度维保，授权费按坐席数
   [星云智能产品手册.md · 数据与安全] ## 数据与安全 - 公有云客户数据在传输与存储两侧均加密，不同租户之间逻辑
② 判官分诊： {"quality": "correct", "reason": "片段[2]明确给出了私有化部署的最低硬件配置：最小8核16线程/32GB内存/200GB SSD，且该片段专门回答‘私有化部署要准备什么服务器’这一问题，信息直接、完整、准确。"}
③ 分支【正确】→ 片段直接用，直接生成（不纠偏）
答案： 根据资料[2]（星云客服FAQ.md · 部署相关）中明确说明：

> **Q：私有化部署要准备什么服务器？**
> A：最小 8 核 16 线程 / 32 GB 内存 / 200 GB SSD，需要 Docker 环境；知识库超过 100 万字建议 64 GB 内存。

该条目针对的是“私有化部署”的通用最低硬件配置要求，并未区分版本（如标准版、企业版）。资料[1][3]均指出企业版“标配私有化部署”，但**未另行规定企业版有更高或不同的最低硬件要求**。因此，企业版私有化部署适用同一套最低配置标准。

✅ 答案：**8 核 16 线程 / 32 GB 内存 / 200 GB SSD，需 Docker 环境**。
========================================================================
问题： 星云智能客服机器人对比其他厂商有什么优势？
① 真实召回 4 条：
   [星云智能产品手册.md · 星云智能客服机器人产品手册] 星云智能客服机器人是星云智能科技公司推出的一款基于大模型的智能客服产品，支持
   [故障排查.md · 故障排查手册] 本文档汇总星云智能客服机器人的常见故障现象、可能原因与排查步骤。
   [星云智能产品手册.md · 产品概述] ## 产品概述 星云智能客服机器人面向企业的售前咨询、售后支持与内部 IT
   [计费与SLA.md · 计费与 SLA 说明] 本文档说明星云智能客服机器人的套餐价格、计费口径、超量计费规则、退款政策与服
② 判官分诊： {"quality": "ambiguous", "reason": "片段[1]和[3]提到了星云智能客服机器人的技术特点（如两段式链路、答案可溯源、基于企业自有文档）和服务规模（3000家企业），但未明确与其他厂商的对比信息，缺乏直接的竞争优势描述。"}
③ 分支【模糊】→ 补充检索：改写检索词再补一轮，与原片段合并后生成
   改写检索词： {"query": "星云智能客服机器人 竞争优势 对比 阿里云 腾讯云 百度 智能客服", "reason": "加入主流竞品厂商名称和‘竞争优势’‘对比’等明确比较意图的关键词，可精准匹配知识库中可能存在的横向评测或差异化分析内容。"}
   + 补充命中 [星云智能产品手册.md · 星云智能客服机器人产品手册] 星云智能客服机器人是星云智能科技公司推出的一款基于大模型的智能客服产品，支持
   + 补充命中 [星云智能产品手册.md · 产品概述] ## 产品概述 星云智能客服机器人面向企业的售前咨询、售后支持与内部 IT
   + 补充命中 [计费与SLA.md · 计费与 SLA 说明] 本文档说明星云智能客服机器人的套餐价格、计费口径、超量计费规则、退款政策与服
   + 补充命中 [故障排查.md · 故障排查手册] 本文档汇总星云智能客服机器人的常见故障现象、可能原因与排查步骤。
   合并后 4 条（原有 4 + 补充 4）
答案： 知识库中没有相关信息。
========================================================================
问题： 星云机器人支持哪些大模型？
① 真实召回 4 条：
   [星云智能产品手册.md · 星云智能客服机器人产品手册] 星云智能客服机器人是星云智能科技公司推出的一款基于大模型的智能客服产品，支持
   [星云智能产品手册.md · 产品概述] ## 产品概述 星云智能客服机器人面向企业的售前咨询、售后支持与内部 IT
   [API文档.md · 星云开放 API 文档] 本文档描述星云智能客服机器人开放 API 的认证方式、核心接口、限流规则与错
   [故障排查.md · 故障排查手册] 本文档汇总星云智能客服机器人的常见故障现象、可能原因与排查步骤。
② 判官分诊： {"quality": "incorrect", "reason": "所有片段均未提及星云机器人支持的具体大模型名称或类型。"}
③ 分支【错误】→ 改写查询重来：换检索词重新召回，并对新结果重新分诊
   改写检索词： {"query": "星云机器人 兼容 大模型列表 支持的AI模型", "reason": "原问题过于简略，改写后加入‘兼容’‘列表’‘支持的AI模型’等高信息量关键词，更贴合技术文档中常见的表述方式，提升命中知识库中相关配置说明或兼容性表格的概率。"}
   + 重新召回 [星云智能产品手册.md · 星云智能客服机器人产品手册] 星云智能客服机器人是星云智能科技公司推出的一款基于大模型的智能客服产品，支持
   + 重新召回 [星云智能产品手册.md · 产品概述] ## 产品概述 星云智能客服机器人面向企业的售前咨询、售后支持与内部 IT
   + 重新召回 [API文档.md · 星云开放 API 文档] 本文档描述星云智能客服机器人开放 API 的认证方式、核心接口、限流规则与错
   + 重新召回 [计费与SLA.md · 计费与 SLA 说明] 本文档说明星云智能客服机器人的套餐价格、计费口径、超量计费规则、退款政策与服
   重新分诊： {"quality": "incorrect", "reason": "所有片段均未提及星云机器人支持的具体大模型名称或类型。"}
   → 纠偏后仍不达标，走兜底：如实告知知识库无相关信息（不硬答，防幻觉）
答案： 知识库中没有相关信息。
========================================================================
问题： 今天上海的天气怎么样？
① 真实召回 4 条：
   [部署与运维手册.md · 监控与告警] ## 监控与告警 建议采集以下指标并配置告警： - 检索命中率（有结果的请求
   [星云客服FAQ.md · 数据与安全] ## 数据与安全 **Q：我的数据会不会被用于训练？** A：不会。客户数据
   [计费与SLA.md · SLA 服务承诺] ## SLA 服务承诺 | 版本 | 可用性承诺 | 故障响应 | 赔付方式
   [星云客服FAQ.md · 计费相关] ## 计费相关 **Q：怎么收费？** A：按版本订阅，基础版 298 元/
② 判官分诊： {"quality": "incorrect", "reason": "所有片段均未提及上海今日天气相关信息，内容涉及监控告警、数据安全、SLA服务承诺和计费问题，与天气查询完全无关。"}
③ 分支【错误】→ 改写查询重来：换检索词重新召回，并对新结果重新分诊
   改写检索词： {"query": "上海今日天气预报", "reason": "添加'预报'一词更符合气象服务类知识库的常见表述，且明确时间（今日）和地点（上海），提升与天气信息片段的匹配度。"}
   + 重新召回 [部署与运维手册.md · 监控与告警] ## 监控与告警 建议采集以下指标并配置告警： - 检索命中率（有结果的请求
   + 重新召回 [计费与SLA.md · SLA 服务承诺] ## SLA 服务承诺 | 版本 | 可用性承诺 | 故障响应 | 赔付方式
   + 重新召回 [API文档.md · 限流规则] ## 限流规则 | 版本 | QPS 上限 | 单请求最大 token |
   + 重新召回 [星云智能产品手册.md · 版本与套餐] 元/月 | 20 | 1000 万字 | 100 | | 企业版 | 按需报
   重新分诊： {"quality": "incorrect", "reason": "所有片段均未提及上海今日天气相关信息，内容涉及监控告警、SLA服务承诺、限流规则及产品套餐，与问题完全无关。"}
   → 纠偏后仍不达标，走兜底：如实告知知识库无相关信息（不硬答，防幻觉）
答案： 知识库中没有相关信息。

→ 本次真实运行覆盖档位： {"correct": 1, "ambiguous": 1, "incorrect": 2}
→ 三档的触发条件就写在上面代码的 if/elif 上：correct=片段足以回答→直接生成；ambiguous=部分相关→改写检索词补一轮再合并；incorrect=片段与问题无关→改写查询重来，重来仍不达标就如实拒答。哪个 query 落哪档由判官现场决定，不伪造分诊结果。""",
             '录制于 2026-09-12，模型 qwen-plus（判官/改写/生成）+ text-embedding-v3 / BM25（检索）')
else:
    _crag = CRAG(k=4)
    _qs = [_crag.run(q) for q in _QUERIES]
    _dist = {k: _qs.count(k) for k in ('correct', 'ambiguous', 'incorrect') if _qs.count(k)}
    print('\n→ 本次真实运行覆盖档位：', _json.dumps(_dist, ensure_ascii=False))
    print('→ 三档的触发条件就写在上面代码的 if/elif 上：correct=片段足以回答→直接生成；'
          'ambiguous=部分相关→改写检索词补一轮再合并；incorrect=片段与问题无关→改写查询重来，'
          '重来仍不达标就如实拒答。哪个 query 落哪档由判官现场决定，不伪造分诊结果。')

## 2. Self-RAG：检索要不要？检索有没有用？

让 LLM 输出**反思标记**（token），自主决策：

```text
[检索决策] 这个问题需要外部知识吗?   → 需要/不需要
[Relevance] 检索到的片段相关吗?       → 相关/无关(丢弃)
[Support]   回答被片段支持吗?        → 有据/无据
[Critique]  整体够不够?              → 继续搜或停止
```

- 需要才检索 → 减少不必要的检索成本；
- 片段无关就丢 → 抑制噪声与幻觉。

In [ ]:
# 知识点·真调说明：Self-RAG 反思标记 —— 让模型自检“片段相关吗？回答被证据支持吗？”
import json as _json
_q = '星云支持私有化部署吗？'
_chunk = '星云机器人提供公有云与私有化两种部署方式；私有化部署需联系销售评估环境。'
_ans = '星云支持私有化部署，且私有化版本附带 7×24 专家驻场与免费硬件扩容。'
print('问题：', _q)
print('检索片段：', _chunk)
print('候选回答：', _ans)
print()
out = _llm_live(
    prompt='问题：' + _q + '\n检索片段：' + _chunk + '\n候选回答：' + _ans +
           '\n请按“反思标记”逐项自检，只输出一个 JSON 对象。',
    system='你是 Self-RAG 反思器。只依据“检索片段”判断候选回答是否有据，不要被候选回答本身说服。'
           '输出 JSON：{"need_retrieval": 布尔, "is_relevant": 布尔, "is_supported": 布尔, '
           '"verdict": "revise / continue / answer 等结论", '
           '"evidence": "指出候选回答中哪句没有被片段支持"}。只输出 JSON 对象，禁止其它文字。',
    fallback='未配置 Key 的固定样例：\n'
             '{"need_retrieval": true, "is_relevant": true, "is_supported": false, '
             '"verdict": "revise", '
             '"evidence": "片段只说提供私有化部署，并未提到 7×24 专家驻场与免费硬件扩容，这两句无据"}',
    temperature=0.2,
)
if out is None:
    out = ('{"need_retrieval": true, "is_relevant": true, "is_supported": false, '
           '"verdict": "revise", '
           '"evidence": "片段只说提供私有化部署，并未提到 7×24 专家驻场与免费硬件扩容，这两句无据"}')
    print('（以上为固定样例；下面用样例走同一条解析）')
try:
    _r = _json.loads(out)
    print('json.loads 通过 ✅ need_retrieval=%s | is_relevant=%s | is_supported=%s' % (
        _r.get('need_retrieval'), _r.get('is_relevant'), _r.get('is_supported')))
    print('verdict：', _r.get('verdict'))
    print('证据缺口：', _r.get('evidence'))
except Exception as _e:
    print('未通过 json.loads：', _e)
print('→ 若直接让模型作答，它可能把“驻场/免费扩容”一并写进答案；Self-RAG 用 [Support]/[Critique] 这类反思标记'
      '拦下“无据”句子再修正——这正是它抑制幻觉的手段。')

## 3. Adaptive RAG：按问题类型路由

```text
问题 → 路由(Router)
   ├─ 简单/直答 → 普通 RAG
   ├─ 复杂多跳 → Query Decomposition / Agent
   ├─ 知识过时 → Web Search / CRAG
   └─ 关系型   → Graph RAG
```

路由判据可以是 **分类器 LLM**（“这是事实题还是综述题”）或规则（关键词/元数据）。



In [ ]:
# 知识点·真调说明：Adaptive RAG 路由 —— 真调模型把问题分诊到最合适的检索/问答方案
import json as _json
_qs = [
    '2025 年销售额最高的商品是哪个？（需查数据库）',
    '通义千问和文心一言现在哪个更强？（观点易过时，需最新资料）',
    '阿里云都收购了哪些公司？各自的 AI 产品线是什么？（跨文档多跳关系）',
    '星云机器人支持私有化部署吗？（查内部产品手册）',
]
print('待路由的问题：')
for _i, _qq in enumerate(_qs, 1):
    print('  %d) %s' % (_i, _qq))
print()
out = _llm_live(
    prompt='为下面 4 个问题各选一个最合适的检索/问答方案，只输出一个 JSON 数组：\n' +
           '\n'.join('%d) %s' % (i + 1, q) for i, q in enumerate(_qs)) +
           '\n可选方案含义：plain_rag=普通向量检索；agentic=多轮拆解后检索；web_search=联网取最新；'
           'graph_rag=图谱关系检索；sql=查数据库。',
    system='你是 Adaptive RAG 的路由器。输出 JSON 数组，每项 {"qid": 数字, "route": "方案名", "reason": "一句话"}。'
           '只依据问题性质选择，禁止输出其它文字。',
    fallback='未配置 Key 的固定样例：\n'
             '[{"qid": 1, "route": "sql", "reason": "数值聚合，答案在数据库"}, '
             '{"qid": 2, "route": "web_search", "reason": "实时对比，静态文档会过时"}, '
             '{"qid": 3, "route": "graph_rag", "reason": "跨实体多跳关系"}, '
             '{"qid": 4, "route": "plain_rag", "reason": "产品手册类，普通向量检索即可"}]',
    temperature=0.1,
)
if out is None:
    out = ('[{"qid": 1, "route": "sql", "reason": "数值聚合，答案在数据库"}, '
           '{"qid": 2, "route": "web_search", "reason": "实时对比，静态文档会过时"}, '
           '{"qid": 3, "route": "graph_rag", "reason": "跨实体多跳关系"}, '
           '{"qid": 4, "route": "plain_rag", "reason": "产品手册类，普通向量检索即可"}]')
    print('（以上为固定样例；下面用样例走同一条解析）')
try:
    _rr = _json.loads(out)
    print('json.loads 通过 ✅ 路由结果：')
    for _x in sorted(_rr, key=lambda z: z['qid']):
        print('  问题%d -> %s（%s）' % (_x['qid'], _x['route'], _x['reason']))
except Exception as _e:
    print('未通过 json.loads：', _e)
print('→ 同一入口按“问题类型”分流：该查库的查库、该联网的联网、该走图的多跳——这就是 Adaptive RAG 的路由器，'
      '也呼应 31 课“文档走 RAG / 数值走 SQL”的并用。')

## 小结

- **CRAG** 事后纠偏（含联网）；**Self-RAG** 事中自省；**Adaptive RAG** 事前路由；
- 三者可组合；共同点：**让模型参与“要不要检索/检索得怎么样”的决策**。